# AutoCompose

## Import Section

In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.chdir("C:/Users/KTS-WS-2501/Documents/Composer-py")
os.getcwd()

'C:\\Users\\KTS-WS-2501\\Documents\\Composer-py'

In [2]:
import torch
torch.manual_seed(64)

In [3]:
import json
import os

In [4]:
from torch.optim import AdamW
from torch.utils.data import Dataset, random_split, DataLoader, RandomSampler, SequentialSampler
from transformers import GPT2LMHeadModel, GPT2Tokenizer, get_linear_schedule_with_warmup

c:\Users\KTS-WS-2501\Documents\composer-py\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## GPU Check

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [38]:
torch.cuda.mem_get_info(device=0)

(0, 6438780928)

In [7]:
print(f"Device name: {torch.cuda.get_device_name(0)}")
props = torch.cuda.get_device_properties(0)
print(f"Total Memory (GB): {props.total_memory / 1024 ** 3}")

Device name: NVIDIA GeForce RTX 4050 Laptop GPU
Total Memory (GB): 5.99658203125


In [39]:
print(f"Allocated: {torch.cuda.memory_allocated() / 1024 ** 3}GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1024 ** 3}GB")

Allocated: 20.29070472717285GB
Reserved: 20.373046875GB


## Load tokenizer and model

In [9]:
model_name = "gpt2"

In [10]:
bos_token = "<|startoftext|>"
eos_token = "<|endoftext|>"
pad_token = "<|pad|>"

In [11]:
# tokenizer = GPT2Tokenizer.from_pretrained(
#     model_name, 
#     bos_token=bos_token, 
#     eos_token=eos_token, 
#     pad_token=pad_token
# )

In [12]:
tokenizer = GPT2Tokenizer.from_pretrained(model_name,bos_token=bos_token,eos_token=eos_token,pad_token=eos_token)
model = GPT2LMHeadModel.from_pretrained(model_name)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 13294.15it/s]


In [13]:
model.resize_token_embeddings(len(tokenizer))

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(50258, 768)

In [14]:
model = model.to(device)

In [15]:
from copy import deepcopy

In [16]:
original_model = deepcopy(model)

## Data Preparation

In [17]:
data_dir = "data"
file_path = "anticipation.json"

In [18]:
file_full_path = os.path.normpath(os.path.join(data_dir,file_path))

In [19]:
with open(file_full_path, "r") as f:
    data = json.load(f)

data[:5]

[{'poem': 'Though the birds sang gayly to him,\nThough the wild-flowers of the meadow\nFilled the air with odors for him;\nThough the forests and the rivers\nSang and shouted at his coming,\nStill his heart was sad within him,\nFor he was alone in heaven.',
  'id': 26},
 {'poem': 'Rise up from your bed of branches,\nRise, O youth, and wrestle with me!"\nFaint with famine, Hiawatha\nStarted from his bed of branches,\nFrom the twilight of his wigwam\nForth into the flush of sunset\nCame, and wrestled with Mondamin;\nAt his touch he felt new courage\nThrobbing in his brain and bosom,\nFelt new life and hope and vigor\nRun through every nerve and fibre.',
  'id': 93},
 {'poem': 'On the morrow and the next day,\nWhen the sun through heaven descending,\nLike a red and burning cinder\nFrom the hearth of the Great Spirit,\nFell into the western waters,\nCame Mondamin for the trial,\nFor the strife with Hiawatha;\nCame as silent as the dew comes,\nFrom the empty air appearing,\nInto empty air r

In [20]:
data[0].keys()

dict_keys(['poem', 'id'])

Size of dataset

In [40]:
print(f"Train samples len: {len(data)}")

Train samples len: 28956


Check dataset

In [42]:
for i in range(3):
    print(data[i])

{'poem': 'Though the birds sang gayly to him,\nThough the wild-flowers of the meadow\nFilled the air with odors for him;\nThough the forests and the rivers\nSang and shouted at his coming,\nStill his heart was sad within him,\nFor he was alone in heaven.', 'id': 26}
{'poem': 'Rise up from your bed of branches,\nRise, O youth, and wrestle with me!"\nFaint with famine, Hiawatha\nStarted from his bed of branches,\nFrom the twilight of his wigwam\nForth into the flush of sunset\nCame, and wrestled with Mondamin;\nAt his touch he felt new courage\nThrobbing in his brain and bosom,\nFelt new life and hope and vigor\nRun through every nerve and fibre.', 'id': 93}
{'poem': 'On the morrow and the next day,\nWhen the sun through heaven descending,\nLike a red and burning cinder\nFrom the hearth of the Great Spirit,\nFell into the western waters,\nCame Mondamin for the trial,\nFor the strife with Hiawatha;\nCame as silent as the dew comes,\nFrom the empty air appearing,\nInto empty air returning,

In [21]:
tokenizer("hello",truncation=True,max_length=10,padding="max_length")

{'input_ids': [31373, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256], 'attention_mask': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

In [22]:
tokenizer.eos_token

'<|endoftext|>'

In [23]:
class PoemDataLoader(Dataset):
    def __init__(self, poems, tokenizer, max_len=768):
        self.attn_mask = []
        self.input_ids = []

        for poem in poems:
            
            encodings_dict = tokenizer("<|startoftext|>"+poem+"<|endoftext|>",max_length=max_len,truncation=True,padding="max_length") 
            self.input_ids.append(torch.tensor(encodings_dict["input_ids"]))
            self.attn_mask.append(torch.tensor(encodings_dict["attention_mask"]))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self,idx):
        return self.input_ids[idx], self.attn_mask[idx]

In [24]:
poems_text = [poem["poem"] for poem in data]
len(poems_text)

28956

In [25]:
poem_data_loader = PoemDataLoader(poems=poems_text,max_len=768,tokenizer=tokenizer)

In [26]:
epochs = 1
warmup_steps = 1e2
sample_every = 100

In [27]:
train_size = int(0.95 * len(data))
val_size = len(data) - train_size

In [28]:
train_data, val_data = random_split(poem_data_loader,[train_size, val_size])

In [29]:
train_data_loader = DataLoader(train_data, batch_size=64, sampler=RandomSampler(train_data))
val_data_loader = DataLoader(val_data, batch_size=64, sampler=SequentialSampler(val_data))

In [30]:
optimizer = AdamW(model.parameters(), lr=5e-4, eps=1e-8)
total_training_steps = len(train_data_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_training_steps)

In [31]:
import time
from datetime import datetime

In [32]:
def format_time(elapsed):
  return str(datetime.timestamp(seconds=int(round(elapsed))))

In [33]:
model.config.vocab_size

50258

In [34]:
print("Tokenizer size:", len(tokenizer))
print("Model vocab size:", model.config.vocab_size)

sample = poem_data_loader.input_ids[0]
print(max(sample))

Tokenizer size: 50258
Model vocab size: 50258
tensor(50257)


In [35]:
for epoch in range(epochs):
    t0 = time.perf_counter()
    total_train_loss = 0
    model.train()
    for step, batch in enumerate(train_data_loader):
        b_input_ids = batch[0].to(device)
        b_labels = batch[0].to(device)
        b_masks = batch[1].to(device)
        
        model.zero_grad()
        print("Max token:", b_input_ids.max())
        print("Min token:", b_input_ids.min())
        print("Vocab size:", model.config.vocab_size)
        outputs = model(b_input_ids, labels=b_labels, attention_mask=b_masks)
        loss = outputs[0]

        batch_loss = loss.item()
        total_train_loss += batch_loss

        if step != 0 and step % sample_every == 0:
            elapsed = format_time(time.perf_counter() - t0)
            print(f"Batch {step} of {len(train_data_loader)}. Loss: {batch_loss}. Time: {elapsed}")

Max token: tensor(50257, device='cuda:0')
Min token: tensor(0, device='cuda:0')
Vocab size: 50258


OutOfMemoryError: CUDA out of memory. Tried to allocate 36.00 MiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 20.29 GiB is allocated by PyTorch, and 84.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

768